Gotta make some tuning curves for homings. \
Two types of plots:
- average tuning for each neuron, with gaussian fit
- single trial activity 
 

Approach:
- smooth data before gaussian fitting?
- fit two peak gaussian?
- plot the individual trials for each cell to see that it is consistent in its firing at each position

In [1]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept, JAL3_22aug

from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_11thSept, JAL4_28aug

from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept, JAL005_5thSept

from behave_analysis.database.Experiments.JAL006_ex import JAL6_flip3_18mar, JAL6_flip7_1apr, JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar

from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr

from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip3_7may, JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may

#JAL3_7sept, JAL3_4sept, JAL3_1sept, JAL3_25aug, JAL3_22aug,
# 
experiments_objects = [JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept,
JAL005_8thSept, JAL005_21stSept, # JAL005_5thSept this one doesn't flip, but can be used as first barrier appearance
JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, # JAL6_flip7_1apr, # this session is sus
JAL7_sesh8_9apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_sesh9_16apr, JAL7_23apr,
JAL8_flip1_25apr,JAL8_flip2_29apr, JAL8_flip3_7may, JAL8_flip4_10may, JAL8_14may]

#
trials = [[1,3],[1,5,7],[1],[2],
    [2,3],[1,3],
    [1,3,4],[1,2,3],[1,2],[1,3],
    [4,5],[1,3],[3,4],[4,5],[2,5],
    [1,3],[1,2],[1,3],[5,7,8],[3,5],
]

In [2]:
%load_ext autoreload
from JR_test_scripts.escape.escape_utils import load, load_homing
from behave_analysis.utils.creating_directories import make_directory
from JR_test_scripts.escape.escape_data_loading_funcs import extract_homing_and_escape_periods
from JR_test_scripts.escape.escape_plotting_funcs import plot_gaussian_fit_tuning
from JR_test_scripts.escape.escape_tuning_funcs import neuron_tuning_by_var, single_trial_tuning, fit_gaussian, fit_double_gaussian

import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
%matplotlib inline

In [29]:
double_wins[11,1]

np.False_

In [16]:
"""Make plots for the neurons using their tuning curves for sorting. Compare to the tuning curves obtained from exploration periods. 
Plot only neurons with xval tuning curves and exclude stationary periods from the analysis"""

%autoreload 2
compression_var = ['y_pos', 'distance_shelter', 'escape', 'speed'] # escape doesn't make sense here, obv
for i, exp in enumerate(experiments_objects[:2]):
    # load data
    session, frame_by_cluster_matrix, behave, y_pos, x_pos, bar, barflip, escape, outofshelter = load(exp)
    ons, offs, homie = load_homing(session, len(behave))
    for comp in compression_var:
        
        nickname = exp.nick_name + '_' + exp.experiment_date + '_' + comp
        dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/"+nickname)
        var, escape_matrix, cond, esc_start, h_start = extract_homing_and_escape_periods(session, frame_by_cluster_matrix, behave, y_pos, x_pos, bar, barflip, comp, ons, offs, no_stationary = False, return_escape = True)

        # eliminate neurons that are all NaN (they probably didn't fire at all during homing/escape)
        bad_neurons = np.all(np.isnan(escape_matrix), axis=1)
        escape_matrix = escape_matrix[~bad_neurons,:]
        
        # compute xval tuning during homing/escape
        peak_firing_condition, tuning, xval = neuron_tuning_by_var(var, escape_matrix, cond, h_start)
        mat_by_cond = single_trial_tuning(escape_matrix, var, cond, h_start)
        y_fitted, R, params, shift_constant, double_wins = plot_gaussian_fit_tuning(tuning, xval, dump_path, mat_by_cond, comp)

2025-01-14 11:53:56.467 | INFO     | behave_analysis.utils.data_loading:load_or_extract_homings:35 - Homings object found. Loading...


Fitted parameters: A = 0.34, mu = 26.67, sigma = 51.78
Fitted parameters (Double Gaussian): A1 = 0.36, mu1 = 57.78, sigma1 = 16.67, A2 = 0.35, mu2 = 7.52, sigma2 = 19.91
Fitted parameters: A = 0.31, mu = 89.00, sigma = 122.06
Fitted parameters (Double Gaussian): A1 = 0.39, mu1 = 60.74, sigma1 = 22.75, A2 = 0.36, mu2 = 5.70, sigma2 = 8.31
Fitted parameters: A = 0.37, mu = 0.00, sigma = 87.48
Fitted parameters (Double Gaussian): A1 = 0.42, mu1 = 5.03, sigma1 = 10.81, A2 = 0.29, mu2 = 89.00, sigma2 = 67.40
Fitted parameters: A = 0.79, mu = 22.00, sigma = 42.58
Fitted parameters (Double Gaussian): A1 = 0.90, mu1 = 23.78, sigma1 = 23.75, A2 = 0.45, mu2 = 74.63, sigma2 = 9.02
Fitted parameters: A = 1.32, mu = 21.78, sigma = 9.86
Fitted parameters (Double Gaussian): A1 = 1.36, mu1 = 21.42, sigma1 = 9.01, A2 = 0.27, mu2 = 63.23, sigma2 = 16.02
Fitted parameters: A = 0.27, mu = 0.00, sigma = 210081.95
Fitted parameters (Double Gaussian): A1 = 0.53, mu1 = 74.87, sigma1 = 8.04, A2 = 0.36, mu2 = 1

2025-01-14 11:59:14.684 | INFO     | behave_analysis.utils.data_loading:load_or_extract_homings:35 - Homings object found. Loading...


Fitted parameters: A = 0.28, mu = 60.42, sigma = 13.33
Fitted parameters (Double Gaussian): A1 = 0.28, mu1 = 60.43, sigma1 = 13.26, A2 = 0.32, mu2 = 0.00, sigma2 = 5.35
Fitted parameters: A = 1.49, mu = 87.00, sigma = 71.11
Fitted parameters: A = 0.25, mu = 31.52, sigma = 26.13
Fitted parameters (Double Gaussian): A1 = 0.17, mu1 = 0.00, sigma1 = 47.33, A2 = 0.15, mu2 = 39.75, sigma2 = 12.42
Fitted parameters: A = 1.46, mu = 46.72, sigma = 25.46
Fitted parameters (Double Gaussian): A1 = 0.50, mu1 = 74.52, sigma1 = 5.67, A2 = 1.48, mu2 = 43.27, sigma2 = 21.84
Fitted parameters: A = 0.09, mu = 0.12, sigma = 5.12
Fitted parameters (Double Gaussian): A1 = 0.08, mu1 = 0.70, sigma1 = 4.85, A2 = 0.01, mu2 = 0.00, sigma2 = 0.89
Fitted parameters: A = 1.74, mu = 87.00, sigma = 41.28
Fitted parameters (Double Gaussian): A1 = 0.82, mu1 = 76.20, sigma1 = 15.88, A2 = 0.99, mu2 = 87.00, sigma2 = 60.03
Fitted parameters: A = 0.29, mu = 0.00, sigma = 10.75
Fitted parameters (Double Gaussian): A1 = 0.07

In [ ]:
"""This fits a single and a double gaussian and plots both"""
from behave_analysis.utils.creating_directories import make_directory
nickname = exp.nick_name + '_' + exp.experiment_date + '_' + comp
dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/"+nickname)

for condition in [0,1,2]:
    xval_true = 1
    test = tuning[condition][xval[:,condition] == xval_true,:]
    for neuron in np.arange(test.shape[0]):
        flatline = np.where(np.diff((np.mean(test, axis = 0) == 0).astype(int)) == 1)[0]
        if len(flatline) > 0:
            test = test[:,:flatline[0]]
        firing_rates  = test[neuron,:]
        firing_rates = firing_rates + abs(np.amin(firing_rates))+ 1e-6 # Add a small epsilon to avoid exact zero
        distances = np.arange(len(firing_rates))

        # Initialize variables
        y_fitted = np.zeros_like(firing_rates)
        R = 0
        y_fitted_double = np.zeros_like(firing_rates)
        R_double = 0
        prominent_peaks = []

        sigma = 4.0  # Standard deviation of the Gaussian kernel
        smoothed_firing_rates = gaussian_filter1d(firing_rates, sigma)

        # Find peaks in firing rates
        peak_indices, _ = find_peaks(smoothed_firing_rates, height=0)  # Only positive peaks

        # Sort peaks by prominence
        if len(peak_indices) > 0:
            sorted_peaks = sorted(peak_indices, key=lambda i: firing_rates[i], reverse=True)
            prominent_peaks = sorted_peaks[0]
            # fit single gaussian
            params = [firing_rates[prominent_peaks], prominent_peaks, np.std(distances)]  # Initial guesses for A, mu, sigma
            bounds = ([0, min(distances), 0], [np.inf, max(distances), np.inf])
            try:
                y_fitted, R, params = fit_gaussian(smoothed_firing_rates, distances, initial_guess = params, constraints = bounds)
            except:
                print("Gaussian fit failed")
            # fit double gaussian
            fit_double = False
            std = np.amin([20,np.amax([10,params[2]])])
            kept_peaks = peak_indices[np.logical_or(peak_indices < prominent_peaks - std, peak_indices > prominent_peaks + std)]
            if len(kept_peaks) > 0:
                sorted_peaks_2 = sorted(kept_peaks, key=lambda i: firing_rates[i], reverse=True)
                prominent_peaks = np.append(prominent_peaks, sorted_peaks_2[0])
                params_double = [firing_rates[prominent_peaks[0]],  # A1
                                prominent_peaks[0],  # mu1 (left peak)
                                np.std(distances),  # sigma1
                                firing_rates[prominent_peaks[1]],  # A1
                                prominent_peaks[1],  # mu2 (right peak)
                                np.std(distances)]   # sigma2
                bounds = ([0, min(distances), 0, 0, min(distances), 0],  # Lower bounds
                        [np.inf, max(distances), np.inf, np.inf, max(distances), np.inf])  # Upper bounds
                        
                try:
                    y_fitted_double, R_double, params_double = fit_double_gaussian(smoothed_firing_rates, distances, initial_guess_double = params_double, constraints = bounds)
                    A1, mu1, sigma1, A2, mu2, sigma2 = params_double
                    # Prominence as relative amplitude
                    A1_relative = A1 / (A1 + A2)
                    A2_relative = A2 / (A1 + A2)

                    # Separation of peaks
                    peak_separation = abs(mu2 - mu1)
                    max_sigma = max(sigma1, sigma2)
                    distinct_peaks = peak_separation > 2 * max_sigma
                    fit_double = True
                except:
                    print("Double Gaussian fit failed")

        # Plot original data and fitted Gaussian
        fig, axs = plt.subplots(1,2,figsize = (12,4))
        axs[0].scatter(distances, firing_rates, label="Original Data", s=3, color="blue")
        axs[0].plot(distances, smoothed_firing_rates, label="Smoothed Data", color="magenta")
        axs[0].plot(distances, y_fitted, label="Fitted Gaussian", color="red")
        axs[0].plot(distances, y_fitted_double, label="Fitted double Gaussian", color="green")
        axs[0].scatter(distances[prominent_peaks], firing_rates[prominent_peaks], color="red", label="Peaks")
        axs[0].set_ylabel("Firing Rate")
        axs[0].set_xlabel(comp)
        axs[0].legend()
        if fit_double:
            axs[0].set_title((f"Gaussian R^2 = {R:.2f}, Double Gaussian R^2 = {R_double:.2f}\n" 
                             f"relative amplitude = {A1_relative:.2f}, {A2_relative:.2f}"))
            if distinct_peaks:
                axs[0].set_title((f"Gaussian R^2 = {R:.2f}, Double Gaussian R^2 = {R_double:.2f}\n" 
                             f"relative amplitude = {A1_relative:.2f}, {A2_relative:.2f}, separate peaks"))
        else:
            axs[0].set_title(f"Gaussian R^2 = {R:.2f}, Double Gaussian R^2 = {R_double:.2f}")

        xval_mat = mat_by_cond[condition][xval[:,condition] == xval_true,:,:]
        axs[1].imshow(xval_mat[neuron,:], cmap="gray_r", vmin = 0, vmax = 1.2, aspect="auto", interpolation = "none")
        axs[1].set_ylabel("Trials")

        c = ['shelter_only', 'barrier', 'barrier_flipped']
        if xval_true:
            fig.savefig(dump_path + "/xval_neuron" + str(neuron) + '_' + c[condition] + ".png")
        else:
            fig.savefig(dump_path + "/neuron" + str(neuron) + '_' + c[condition] + ".png")
        plt.close()

In [ ]:
"""This fits a single and a double gaussian and plots and returns only the chosen model"""
%autoreload 2
from behave_analysis.utils.creating_directories import make_directory
exp = experiments_objects[7]
session, frame_by_cluster_matrix, behave, y_pos, x_pos, bar, barflip, escape, outofshelter = load(exp)
ons, offs, homie = load_homing(session, len(behave))
comp = 'escape'
nickname = exp.nick_name + '_' + exp.experiment_date + '_' + comp
dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/"+nickname)
var, escape_matrix, cond, esc_start, h_start = extract_homing_and_escape_periods(session, frame_by_cluster_matrix, behave, y_pos, x_pos, bar, barflip, comp, ons, offs, no_stationary = False, return_escape = True)

# eliminate neurons that are all NaN (they probably didn't fire at all during homing/escape)
bad_neurons = np.all(np.isnan(escape_matrix), axis=1)
escape_matrix = escape_matrix[~bad_neurons,:]

# compute xval tuning during homing/escape
peak_firing_condition, tuning, xval = neuron_tuning_by_var(var, escape_matrix, cond, h_start)
mat_by_cond = single_trial_tuning(escape_matrix, var, cond, h_start)

y_fitted, R, params, shift_constant, double_wins = plot_gaussian_fit_tuning(tuning, xval, dump_path, mat_by_cond, comp)

Fitted parameters: A = 2.13, mu = 36.99, sigma = 21.82
Fitted parameters: A = 2.28, mu = 17.97, sigma = 62.82
Fitted parameters (Double Gaussian): A1 = 2.40, mu1 = 6.03, sigma1 = 34.24, A2 = 1.63, mu2 = 69.63, sigma2 = 17.61
Fitted parameters: A = 0.29, mu = 0.00, sigma = 63.08
Fitted parameters (Double Gaussian): A1 = 0.12, mu1 = 0.00, sigma1 = 15.91, A2 = 0.22, mu2 = 0.00, sigma2 = 84.70
Fitted parameters: A = 0.58, mu = 0.00, sigma = 47.73
Fitted parameters (Double Gaussian): A1 = 0.48, mu1 = 10.17, sigma1 = 11.72, A2 = 0.23, mu2 = 100.00, sigma2 = 131639.96
Fitted parameters: A = 0.13, mu = 54.21, sigma = 31.26
Fitted parameters (Double Gaussian): A1 = 0.15, mu1 = 58.40, sigma1 = 22.39, A2 = 0.19, mu2 = 0.00, sigma2 = 5.07
Fitted parameters: A = 1.52, mu = 9.07, sigma = 30.55
Fitted parameters (Double Gaussian): A1 = 1.52, mu1 = 10.61, sigma1 = 28.85, A2 = 0.21, mu2 = 96.57, sigma2 = 8.25
Fitted parameters: A = 1.57, mu = 0.00, sigma = 52.57
Fitted parameters (Double Gaussian): A1 